## silver

In [0]:
from pyspark.sql.functions import col
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, functions as f
from functools import reduce

### schema creation

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalogmeteo.silver;

take the tables from the bronze layer

In [0]:
weather_stations = spark.read.table(f"catalogmeteo.bronze.weather_stations_metadata")

## fix the schema

### for every table that involves observations such that negative temperatures are read properly

In [0]:
from pyspark.sql import functions as F

extract_pattern = r"(-?[0-9]+\.?[0-9]*)"
numeric_string_cols = {"temperature_c", "feels_like_c"}

database_name = "catalogmeteo.bronze"

tabelle_reali = [
    t.name for t in spark.catalog.listTables(database_name) 
    if t.tableType != 'TEMPORARY'
]
tabelle_clean = [t for t in tabelle_reali if "stations" not in t.lower()]

dfs_to_union = []

for t_name in tabelle_clean:
    df = spark.table(f"{database_name}.{t_name}")
    
    target_cols = [c for c, t in df.dtypes if t == "string" and c in numeric_string_cols]
    
    for col_name in target_cols:
        df = df.withColumn(
            col_name,
            F.regexp_extract(
                F.regexp_replace(F.col(col_name), "\u2212", "-"),  # − → -
                extract_pattern, 1
            ).cast("double")
        )
    
    dfs_to_union.append(df)

### union of observations and drop irrelevant columns

In [0]:
if dfs_to_union:
    # Register each DataFrame as a temp view
    for i, df in enumerate(dfs_to_union):
        df.createOrReplaceTempView(f"weather_temp_{i}")

    # Build the UNION query (SQL UNION removes duplicates by default)
    union_query = "\nUNION\n".join([
        f"SELECT * FROM weather_temp_{i}" 
        for i in range(len(dfs_to_union))
    ])

    weather_final_union = spark.sql(union_query).drop("uv_index", "_rescued_data")


In [0]:
weather_final_union.count()

121

In [0]:
weather_stations = weather_stations.drop("_rescued_data")

## Upsert: save the united observations in the silver layer

In [0]:
from delta.tables import DeltaTable

table_name = "catalogmeteo.silver.weather_observations"
delta_table = DeltaTable.forName(spark, table_name)

# Delete ALL rows for keys that have duplicates
delta_table.delete(
    "CONCAT(station_id, '_', date) IN (SELECT CONCAT(station_id, '_', date) FROM catalogmeteo.silver.weather_observations GROUP BY station_id, date HAVING COUNT(*) > 1)"
)

# Then re-insert the correct deduplicated version of those rows from source
(
    delta_table.alias("target")
    .merge(
        weather_final_union.alias("source"),
        "target.station_id = source.station_id AND target.date = source.date"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:

if spark.catalog.tableExists(table_name):
    delta_table = DeltaTable.forName(spark, table_name)
    (
        delta_table.alias("target")
        .merge(
            weather_final_union.alias("source"),
            "target.station_id = source.station_id AND target.date = source.date"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    weather_final_union.write.format("delta").saveAsTable(table_name)

In [0]:
spark.sql("select * from catalogmeteo.silver.weather_observations where date = '2025-01-17' and station_id = 'MI002'").display()

date,city,country,temperature_c,feels_like_c,humidity_pct,wind_speed_kmh,wind_direction,precipitation_mm,weather_condition,visibility_km,pressure_hpa,station_id
2025-01-17,Milan,IT,4.8,1.5,80,14.2,NW,3.6,Foggy,3.1,1008.4,MI002


In [0]:
# unified observations + station info
table_name = "catalogmeteo.silver.weather_stations"

if spark.catalog.tableExists(table_name):
    delta_table = DeltaTable.forName(spark, table_name)
    (
        delta_table.alias("target")
        .merge(
            weather_stations.alias("source"),
            "target.station_id = source.station_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    weather_stations.write.format("delta").saveAsTable(table_name)

### non ha senso salvare queste tabelle silver in adlls perché ci sono già nello unity catalog e on servono come external tables per applicazioni esterne


### ottimizzazione tabelle

In [0]:
%sql
OPTIMIZE catalogmeteo.silver.weather_observations ZORDER BY station_id;

path,metrics
abfss://source@storageaccountmeteo.dfs.core.windows.net/__unitystorage/catalogs/c9ab5bf7-9cb7-429e-86b9-01f970210f5b/tables/2043baec-fb2c-45cf-82f8-d7c4e85c283f,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 6444), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1775122990193, 1775122990738, 8, 0, null, List(0, 0), null, 13, 13, 0, 0, null, null)"


In [0]:
%sql
OPTIMIZE catalogmeteo.silver.weather_stations ZORDER BY station_id;

path,metrics
abfss://source@storageaccountmeteo.dfs.core.windows.net/__unitystorage/catalogs/c9ab5bf7-9cb7-429e-86b9-01f970210f5b/tables/0494fcd9-61af-4369-be0b-97043cdb9d13,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 5204), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1775122999310, 1775122999679, 8, 0, null, List(0, 0), null, 16, 16, 0, 0, null, null)"
